# 🎬 Murano Vidraçaria — Geração de Vídeo Ultrarrealista com Wan 2.1 / 2.2 (Image-to-Video)
**GPU Gratuita no Kaggle (30h/semana — Selecione `GPU T4 x2` nas configurações à direita)**

### Como usar este Notebook:
1. No menu lateral direito do Kaggle (**Settings -> Accelerator**), selecione **GPU T4 x2**.
2. Faça upload da foto real da varanda da Murano (`obra-varanda-gourmet.jpg` ou `obra-envidracamento-sacada.png`) clicando em **Upload -> New Dataset**.
3. Clique em **Run All** para instalar o `diffusers` atualizado e gerar o vídeo `.mp4` fotorrealista das folhas de vidro deslizando e abrindo a 90°.

In [ ]:
# 1. Instalação das dependências oficiais do Wan Video (HuggingFace Diffusers + Accelerate)
!pip install -q git+https://github.com/huggingface/diffusers.git transformers accelerate imageio-ffmpeg ftfy

In [ ]:
# 2. Carregamento do Modelo Wan2.1-I2V (Otimizado em FP16 com CPU Offload para caber com folga na T4 de 16GB)
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image
from transformers import CLIPVisionModel

model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
image_encoder = CLIPVisionModel.from_pretrained(model_id, subfolder="image_encoder", torch_dtype=torch.float32)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(
    model_id,
    vae=vae,
    image_encoder=image_encoder,
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
print("Modelo Wan I2V carregado com sucesso na GPU do Kaggle!")

In [ ]:
# 3. Geração do Vídeo Ultrarrealista a partir da Foto Real da Murano Vidraçaria
# Altere o caminho abaixo para o caminho da foto enviada no seu Kaggle Dataset:
IMAGE_PATH = "/kaggle/input/murano-fotos/obra-varanda-gourmet.jpg"

image = load_image(IMAGE_PATH)

prompt = (
    "Photorealistic architectural cinematography, 8k luxury apartment balcony in Brazil. "
    "Frameless clear tempered glass panels smoothly sliding along the black anodized aluminum rail "
    "and pivoting 90 degrees open to the left side. Natural sunlight reflections on the glass edges, "
    "smooth steady camera, ultra-realistic architectural digest video."
)
negative_prompt = "cartoon, 3d render, CGI, low resolution, blurry, warped geometry, deformed glass, watermark"

output = pipe(
    image=image,
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=480,
    width=832,
    num_frames=81,
    guidance_scale=5.0
).frames[0]

export_to_video(output, "/kaggle/working/sacada_wan22.mp4", fps=16)
print("Vídeo gerado com sucesso em /kaggle/working/sacada_wan22.mp4! Baixe e salve em assets/videos/sacada_wan22.mp4")